In [1]:
"""Train the COPOD detector and scale its anomaly scores."""

import os
import sys

# Clear any cached modules
if 'config' in sys.modules:
    del sys.modules['config']
if 'to_tflite' in sys.modules:
    del sys.modules['to_tflite']

import pandas as pd
from pyod.models.copod import COPOD
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import config
import to_tflite

# Build the detector from its configuration.
copod = COPOD(
    contamination=config.CONTAMINATION,
    n_jobs=config.N_JOBS,
)

# NOTE: all dataset features must be pre-normalized so that, after the int8
# quantization in keras/to_tflite.py, no value overflows the int8 range.
# Fit the configured features from the CSV.
features = pd.read_csv(config.CSV_PATH)[config.FEATURES].to_numpy()
X_train, X_test = train_test_split(
    features, test_size=config.TEST_SIZE, random_state=config.RANDOM_STATE
)
copod.fit(X_train)

# Scale the raw anomaly scores into the configured range.
raw_scores = copod.decision_function(X_train).reshape(-1, 1)
score_scaler = MinMaxScaler(feature_range=config.SCORE_RANGE).fit(raw_scores)
train_scores = score_scaler.transform(raw_scores).ravel()
test_scores = score_scaler.transform(
    copod.decision_function(X_test).reshape(-1, 1)
).ravel()

# Learn the scores with a small network and export .keras + .tflite.
to_tflite.to_tflite(
    name="copod",
    x_train=X_train,
    y_train=train_scores,
    x_test=X_test,
    y_test=test_scores,
    layers=config.LAYERS,
    learning_rate=config.LEARNING_RATE,
    epochs=config.EPOCHS,
    batch_size=config.BATCH_SIZE,
    verbose=config.VERBOSE,
    feature_names=config.FEATURES,
)


2026-08-14 18:37:58.677615: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Epoch 1/50


2026-08-14 18:38:04.810836: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


3747/3747 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - loss: 2.8833e-04 - val_loss: 1.7605e-04
Epoch 2/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 1.4442e-04 - val_loss: 1.6420e-04
Epoch 3/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 1.1378e-04 - val_loss: 8.2403e-05
Epoch 4/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - loss: 7.8544e-05 - val_loss: 7.1376e-05
Epoch 5/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - loss: 7.3850e-05 - val_loss: 7.7421e-05
Epoch 6/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - loss: 6.9381e-05 - val_loss: 7.7602e-05
Epoch 7/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 6.5834e-05 - val_loss: 5.7013e-05
Epoch 8/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - loss: 6.3772e-05 - val_loss: 9.9472e-05
Epoch 9/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 5.8781e-05 - val_loss: 5.2376e-05
Epoch 10/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - loss: 5.6641e-05 - val_loss: 5.1001e-05
Epoch 11/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms

INFO:tensorflow:Assets written to: /tmp/tmp_u0zdg1b/assets


Saved artifact at '/tmp/tmp_u0zdg1b'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 2), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  139824830624976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139824830627664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139824827152912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139824827153296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139824827152720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139824827151568: TensorSpec(shape=(), dtype=tf.resource, name=None)


/home/patrik/Diplomka/LODA/env/lib/python3.12/site-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1786725869.268261   29840 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1786725869.268691   29840 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
2026-08-14 18:44:29.270005: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmp_u0zdg1b
2026-08-14 18:44:29.273044: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-08-14 18:44:29.273051: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmp_u0zdg1b
I0000 00:00:1786725869.275495   29840 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
2026-08-14 18:44:29.275830: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-08-14 1

<Sequential name=sequential, built=True>